# SOLETE Cross-Resolution Comparison

This notebook compares the four temporal resolutions available in the
SOLETE dataset:

- 60-minute
- 5-minute
- 1-minute
- 1-second

The individual resolutions have already been examined separately in:

- `01_eda_60min.ipynb`
- `02_eda_5min.ipynb`
- `03_eda_1min.ipynb`
- `04_eda_1sec.ipynb`

Therefore, this notebook does **not repeat the full exploratory data
analysis**.

## Objectives

The purpose of this comparison is to determine how temporal resolution
affects the structure and behaviour of the SOLETE data and to select the
most appropriate resolution for the main forecasting experiments.

The comparison will focus on:

1. dataset size and temporal coverage;
2. timestamp structure and chronological ordering;
3. sampling interval;
4. missing and duplicate observations;
5. target-variable behaviour:
   - `P_Solar[kW]`
   - `P_Gaia[kW]`;
6. important predictor relationships;
7. data-quality issues such as:
   - pressure artifacts,
   - wind-direction representation,
   - midnight boundary effects,
   - sparse variables in the 1-second data;
8. information content versus computational cost;
9. realistic suitability for time-series forecasting.

## Important Methodological Principle

The temporal resolution will **not** be selected simply because one
resolution contains more observations.

The selected resolution should provide a useful balance between:

- temporal detail,
- data quality,
- physical interpretability,
- computational efficiency,
- forecasting realism,
- and reproducibility.

The original SOLETE measurements will remain unchanged during this
comparison.

Any preprocessing or anomaly-treatment decisions will be considered
only after the forecasting resolution has been selected.

In [1]:
# ============================================================
# CELL 1 — PROJECT SETUP AND SOLETE DATASET PATHS
# ============================================================

from pathlib import Path
import sys

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PROJECT ROOT
# ------------------------------------------------------------
#
# This notebook is stored inside:
#
#     Project/notebooks/
#
# We identify the project root in a way that also works if
# the notebook is launched from the project root itself.

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


print("Project root:")
print(PROJECT_ROOT)


# ------------------------------------------------------------
# ADD PROJECT ROOT TO PYTHON IMPORT PATH
# ------------------------------------------------------------

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


# Import the reusable SOLETE loader.
from src.data_loader import load_solete


# ------------------------------------------------------------
# DATASET PATHS
# ------------------------------------------------------------

dataset_paths = {
    "60min": (
        PROJECT_ROOT
        / "solete_dataset"
        / "SOLETE_Pombo_60min.h5"
    ),

    "5min": (
        PROJECT_ROOT
        / "solete_dataset"
        / "SOLETE_Pombo_5min.h5"
    ),

    "1min": (
        PROJECT_ROOT
        / "solete_dataset"
        / "SOLETE_Pombo_1min.h5"
    ),

    "1sec": (
        PROJECT_ROOT
        / "solete_dataset"
        / "SOLETE_Pombo_1sec.h5"
    ),
}


# ------------------------------------------------------------
# VERIFY THAT ALL FOUR FILES EXIST
# ------------------------------------------------------------

print("\nDataset files:")

for resolution, path in dataset_paths.items():

    print(
        f"{resolution:>5} : "
        f"{'FOUND' if path.exists() else 'MISSING'}"
    )

    print(f"        {path}")

Project root:
/Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project

Dataset files:
60min : FOUND
        /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_60min.h5
 5min : FOUND
        /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_5min.h5
 1min : FOUND
        /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_1min.h5
 1sec : FOUND
        /Users/siddharthgupta/Downloads/NSUT/Semester 1/MTITC102 AI and Machine Learning/Project/solete_dataset/SOLETE_Pombo_1sec.h5


In [2]:
# ============================================================
# CELL 2 — LOAD 60-MIN, 5-MIN, AND 1-MIN DATASETS
# ============================================================

# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The 60-minute, 5-minute, and 1-minute datasets are small
# enough to load comfortably into pandas.
#
# The 1-second dataset contains ~39.5 million observations,
# so it will continue to be handled separately using HDF5
# chunked reads.
#
# IMPORTANT:
#
# The original 60-minute file is not stored chronologically.
# Therefore:
#
#   - df_60min_raw preserves the source row order
#   - df_60min is a chronologically sorted copy
#
# The 5-minute and 1-minute datasets were already confirmed
# to be chronologically ordered.


# ------------------------------------------------------------
# LOAD DATASETS
# ------------------------------------------------------------

df_60min_raw = load_solete(
    dataset_paths["60min"]
)

df_5min = load_solete(
    dataset_paths["5min"]
)

df_1min = load_solete(
    dataset_paths["1min"]
)


# ------------------------------------------------------------
# CREATE CHRONOLOGICAL 60-MINUTE COPY
# ------------------------------------------------------------

df_60min = (
    df_60min_raw
    .sort_index()
)


# ------------------------------------------------------------
# STORE THE THREE MANAGEABLE RESOLUTIONS TOGETHER
# ------------------------------------------------------------
#
# This dictionary will make later comparisons easier.

dataframes = {
    "60min": df_60min,
    "5min": df_5min,
    "1min": df_1min,
}


# ------------------------------------------------------------
# DISPLAY BASIC CONFIRMATION
# ------------------------------------------------------------

for resolution, dataframe in dataframes.items():

    print(f"\n{resolution}")
    print("-" * 40)

    print("Shape:")
    print(dataframe.shape)

    print("Start:")
    print(dataframe.index.min())

    print("End:")
    print(dataframe.index.max())

    print("Chronological:")
    print(dataframe.index.is_monotonic_increasing)


60min
----------------------------------------
Shape:
(10969, 11)
Start:
2018-06-01 00:00:00+00:00
End:
2019-09-01 00:00:00+00:00
Chronological:
True

5min
----------------------------------------
Shape:
(131617, 11)
Start:
2018-06-01 00:00:00+00:00
End:
2019-09-01 00:00:00+00:00
Chronological:
True

1min
----------------------------------------
Shape:
(658081, 11)
Start:
2018-06-01 00:00:00+00:00
End:
2019-09-01 00:00:00+00:00
Chronological:
True


In [3]:
# ============================================================
# CELL 3 — CROSS-RESOLUTION STRUCTURAL SUMMARY
# ============================================================

import h5py
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Compare the structural properties of all four SOLETE
# resolutions.
#
# IMPORTANT:
#
# We distinguish between:
#
#   1. SOURCE ORDER
#      The order in which observations are physically stored
#      in the original HDF5 file.
#
#   2. ANALYSIS ORDER
#      Whether the data are chronologically ordered after
#      preparation for time-series analysis.
#
# This distinction is important because:
#
#   - the original 60-minute file is not chronologically
#     stored, although we can sort it easily;
#
#   - the 1-second file is stored as shuffled daily blocks,
#     even though its complete timestamp grid is unique and
#     complete.


# ------------------------------------------------------------
# EXPECTED SAMPLING INTERVALS
# ------------------------------------------------------------

expected_intervals = {
    "60min": pd.Timedelta(hours=1),
    "5min": pd.Timedelta(minutes=5),
    "1min": pd.Timedelta(minutes=1),
}


# ------------------------------------------------------------
# SOURCE DATAFRAMES
# ------------------------------------------------------------
#
# For 60-minute data we preserve both:
#
#   df_60min_raw -> original HDF5 row order
#   df_60min     -> chronologically sorted copy

source_dataframes = {
    "60min": df_60min_raw,
    "5min": df_5min,
    "1min": df_1min,
}

analysis_dataframes = {
    "60min": df_60min,
    "5min": df_5min,
    "1min": df_1min,
}


# ------------------------------------------------------------
# BUILD SUMMARY FOR 60-MIN, 5-MIN, AND 1-MIN
# ------------------------------------------------------------

summary_rows = []


for resolution in [
    "60min",
    "5min",
    "1min",
]:

    source_df = source_dataframes[resolution]
    analysis_df = analysis_dataframes[resolution]

    expected_interval = (
        expected_intervals[resolution]
    )


    # --------------------------------------------------------
    # SOURCE-ORDER INTERVAL CHECK
    # --------------------------------------------------------

    source_differences = (
        source_df.index
        .to_series()
        .diff()
        .dropna()
    )

    source_unexpected_intervals = (
        source_differences
        != expected_interval
    ).sum()


    # --------------------------------------------------------
    # ANALYSIS-ORDER INTERVAL CHECK
    # --------------------------------------------------------

    analysis_differences = (
        analysis_df.index
        .to_series()
        .diff()
        .dropna()
    )

    analysis_unexpected_intervals = (
        analysis_differences
        != expected_interval
    ).sum()


    summary_rows.append({

        "resolution":
            resolution,

        "rows":
            len(source_df),

        "columns":
            source_df.shape[1],

        "start":
            analysis_df.index.min(),

        "end":
            analysis_df.index.max(),

        "missing_values":
            source_df.isna().sum().sum(),

        "duplicate_timestamps":
            source_df.index.duplicated().sum(),

        "source_order_chronological":
            source_df.index.is_monotonic_increasing,

        "source_order_unexpected_intervals":
            source_unexpected_intervals,

        "analysis_order_chronological":
            analysis_df.index.is_monotonic_increasing,

        "analysis_order_unexpected_intervals":
            analysis_unexpected_intervals,

        "timestamp_grid_complete":
            (
                analysis_unexpected_intervals == 0
                and
                analysis_df.index.is_unique
            ),
    })


# ------------------------------------------------------------
# ADD 1-SECOND DATASET
# ------------------------------------------------------------

with h5py.File(
    dataset_paths["1sec"],
    "r"
) as f:

    values_ds = (
        f["DATA"]["block0_values"]
    )

    one_sec_rows = (
        values_ds.shape[0]
    )

    one_sec_columns = (
        values_ds.shape[1]
    )


# Results already established during the dedicated
# 1-second EDA:
#
#   - 39,484,801 rows
#   - 456 non-1-second transitions in stored order
#   - 0 global duplicate timestamps
#   - 0 missing expected timestamps
#   - exact complete 1-second grid
#   - stored order is not chronological

summary_rows.append({

    "resolution":
        "1sec",

    "rows":
        one_sec_rows,

    "columns":
        one_sec_columns,

    "start":
        pd.Timestamp(
            "2018-06-01 00:00:00",
            tz="UTC"
        ),

    "end":
        pd.Timestamp(
            "2019-09-01 00:00:00",
            tz="UTC"
        ),

    "missing_values":
        0,

    "duplicate_timestamps":
        0,

    "source_order_chronological":
        False,

    "source_order_unexpected_intervals":
        456,

    # The complete timestamp grid can be restored into
    # chronological order without gaps or duplicates.
    "analysis_order_chronological":
        True,

    "analysis_order_unexpected_intervals":
        0,

    "timestamp_grid_complete":
        True,
})


# ------------------------------------------------------------
# CREATE FINAL STRUCTURAL COMPARISON TABLE
# ------------------------------------------------------------

structural_summary = (
    pd.DataFrame(summary_rows)
    .set_index("resolution")
)

structural_summary

,rows,columns,start,end,missing_values,duplicate_timestamps,source_order_chronological,source_order_unexpected_intervals,analysis_order_chronological,analysis_order_unexpected_intervals,timestamp_grid_complete
resolution,,,,,,,,,,,
60min,10969,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,False,456,True,0,True
5min,131617,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,True,0,True,0,True
1min,658081,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,True,0,True,0,True
1sec,39484801,11,2018-06-01 00:00:00+00:00,2019-09-01 00:00:00+00:00,0,0,False,456,True,0,True


In [4]:
# ============================================================
# CELL 4 — CROSS-RESOLUTION TARGET-VARIABLE SUMMARY
# ============================================================

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Compare the two renewable-energy target variables across
# all four temporal resolutions:
#
#     P_Solar[kW]
#     P_Gaia[kW]
#
# We compare:
#
#   - number of observations
#   - mean
#   - standard deviation
#   - minimum
#   - maximum
#   - number of zero observations
#   - percentage of zero observations
#   - number of non-zero observations
#
# For:
#
#   60-minute
#   5-minute
#   1-minute
#
# we can use the pandas DataFrames already in memory.
#
# For:
#
#   1-second
#
# we calculate the statistics directly from the HDF5 file
# in chunks so that ~39.5 million rows are never loaded
# into pandas at once.


# ------------------------------------------------------------
# TARGET VARIABLES
# ------------------------------------------------------------

target_variables = [
    "P_Solar[kW]",
    "P_Gaia[kW]",
]


# ------------------------------------------------------------
# STORAGE FOR RESULTS
# ------------------------------------------------------------

target_summary_rows = []


# ============================================================
# PART A — 60-MIN, 5-MIN, AND 1-MIN DATA
# ============================================================

for resolution, dataframe in dataframes.items():

    for target in target_variables:

        series = dataframe[target]

        zero_count = (
            series == 0
        ).sum()

        nonzero_count = (
            series != 0
        ).sum()

        target_summary_rows.append({

            "resolution":
                resolution,

            "target":
                target,

            "count":
                len(series),

            "mean":
                series.mean(),

            "std":
                series.std(),

            "min":
                series.min(),

            "max":
                series.max(),

            "zero_count":
                zero_count,

            "zero_percent":
                zero_count
                / len(series)
                * 100,

            "nonzero_count":
                nonzero_count,
        })


# ============================================================
# PART B — 1-SECOND DATA
# ============================================================

# Column positions in the HDF5 measurement matrix.
#
# From the SOLETE column order:
#
#   P_Gaia  -> column 6
#   P_Solar -> column 7

one_sec_target_columns = {
    "P_Solar[kW]": 7,
    "P_Gaia[kW]": 6,
}


chunk_size = 1_000_000


with h5py.File(
    dataset_paths["1sec"],
    "r"
) as f:

    values_ds = (
        f["DATA"]["block0_values"]
    )

    total_rows = (
        values_ds.shape[0]
    )


    for target, column_index in (
        one_sec_target_columns.items()
    ):

        # ----------------------------------------------------
        # RUNNING STATISTICS
        # ----------------------------------------------------

        count = 0

        value_sum = 0.0
        value_sum_squared = 0.0

        minimum = np.inf
        maximum = -np.inf

        zero_count = 0


        # ----------------------------------------------------
        # SCAN THE COMPLETE COLUMN
        # ----------------------------------------------------

        for start in range(
            0,
            total_rows,
            chunk_size
        ):

            end = min(
                start + chunk_size,
                total_rows
            )

            values = values_ds[
                start:end,
                column_index
            ]


            count += len(values)

            value_sum += (
                values.sum()
            )

            value_sum_squared += (
                np.square(values).sum()
            )

            minimum = min(
                minimum,
                values.min()
            )

            maximum = max(
                maximum,
                values.max()
            )

            zero_count += (
                values == 0
            ).sum()


        # ----------------------------------------------------
        # CALCULATE EXACT MEAN AND SAMPLE STANDARD DEVIATION
        # ----------------------------------------------------
        #
        # pandas Series.std() uses sample standard deviation:
        #
        #     ddof = 1
        #
        # so we calculate the same quantity here for a fair
        # cross-resolution comparison.

        mean = (
            value_sum
            / count
        )

        variance = (
            value_sum_squared
            - count * mean**2
        ) / (
            count - 1
        )

        std = np.sqrt(
            max(variance, 0)
        )

        nonzero_count = (
            count - zero_count
        )


        target_summary_rows.append({

            "resolution":
                "1sec",

            "target":
                target,

            "count":
                count,

            "mean":
                mean,

            "std":
                std,

            "min":
                minimum,

            "max":
                maximum,

            "zero_count":
                zero_count,

            "zero_percent":
                zero_count
                / count
                * 100,

            "nonzero_count":
                nonzero_count,
        })


# ============================================================
# PART C — CREATE COMPARISON TABLE
# ============================================================

target_summary = (
    pd.DataFrame(
        target_summary_rows
    )
    .set_index(
        [
            "target",
            "resolution"
        ]
    )
    .sort_index()
)


target_summary

count      mean       std  min        max  \
target      resolution                                                 
P_Gaia[kW]  1min          658081  2.691753  3.434584  0.0  23.478949   
            1sec        39484801  0.016261  0.334323  0.0  12.313000   
            5min          131617  2.694643  3.382534  0.0  23.330538   
            60min          10969  0.016259  0.323335  0.0  10.184867   
P_Solar[kW] 1min          658081  0.988283  1.688230  0.0   8.404550   
            1sec        39484801  1.024471  1.738146  0.0   9.831133   
            5min          131617  0.987980  1.666584  0.0   7.715423   
            60min          10969  1.029960  1.657731  0.0   6.975178   

                        zero_count  zero_percent  nonzero_count  
target      resolution                                           
P_Gaia[kW]  1min            225913     34.329057         432168  
            1sec          39349911     99.658375         134890  
            5min             38812     29.488592          92805  
            60min            10921     99.562403             48  
P_Solar[kW] 1min            272444     41.399767         385637  
            1sec          16758568     42.443086       22726233  
            5min             54303     41.258348          77314  
            60min             4204     38.326192           6765

In [5]:
# ============================================================
# CELL 5 — TEST CROSS-RESOLUTION TARGET CONSISTENCY
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 4 showed:
#
# SOLAR:
#   P_Solar behaves similarly across all resolutions.
#
# WIND:
#   P_Gaia behaves similarly for:
#
#       1-second <-> 60-minute
#
#   but very differently for:
#
#       1-minute <-> 5-minute
#
# We therefore test whether the provided coarser-resolution
# target variables can be reproduced by simple averaging of
# the next finer resolution.
#
# Specifically:
#
#   1-minute -> 5-minute
#   5-minute -> 60-minute
#
# We examine:
#
#   - P_Solar[kW]
#   - P_Gaia[kW]
#
# IMPORTANT:
# This is a diagnostic comparison only.
# No source dataset is modified.


# ------------------------------------------------------------
# 1. AGGREGATE 1-MINUTE DATA TO 5-MINUTE RESOLUTION
# ------------------------------------------------------------

targets = [
    "P_Solar[kW]",
    "P_Gaia[kW]",
]

from_1min_to_5min = (
    df_1min[targets]
    .resample("5min")
    .mean()
)


# ------------------------------------------------------------
# 2. AGGREGATE 5-MINUTE DATA TO 60-MINUTE RESOLUTION
# ------------------------------------------------------------

from_5min_to_60min = (
    df_5min[targets]
    .resample("60min")
    .mean()
)


# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def compare_target_series(
    provided,
    reconstructed,
    label
):
    """
    Compare a target variable from a provided SOLETE
    resolution with the same variable reconstructed by
    averaging a finer-resolution dataset.
    """

    # Align on common timestamps.
    comparison = pd.concat(
        [
            provided.rename("provided"),
            reconstructed.rename("reconstructed"),
        ],
        axis=1,
        join="inner",
    ).dropna()

    difference = (
        comparison["provided"]
        -
        comparison["reconstructed"]
    )

    absolute_difference = difference.abs()

    print(f"\n{label}")
    print("-" * 60)

    print("Number of aligned timestamps:")
    print(len(comparison))

    print("\nMean absolute difference:")
    print(absolute_difference.mean())

    print("\nMedian absolute difference:")
    print(absolute_difference.median())

    print("\nMaximum absolute difference:")
    print(absolute_difference.max())

    print("\nCorrelation:")
    print(
        comparison[
            ["provided", "reconstructed"]
        ].corr().iloc[0, 1]
    )


# ------------------------------------------------------------
# 3. COMPARE PROVIDED 5-MIN DATA WITH 1-MIN -> 5-MIN
# ------------------------------------------------------------

for target in targets:

    compare_target_series(
        provided=df_5min[target],
        reconstructed=from_1min_to_5min[target],
        label=f"{target}: provided 5min vs 1min averaged to 5min",
    )


# ------------------------------------------------------------
# 4. COMPARE PROVIDED 60-MIN DATA WITH 5-MIN -> 60-MIN
# ------------------------------------------------------------

for target in targets:

    compare_target_series(
        provided=df_60min[target],
        reconstructed=from_5min_to_60min[target],
        label=f"{target}: provided 60min vs 5min averaged to 60min",
    )


P_Solar[kW]: provided 5min vs 1min averaged to 5min
------------------------------------------------------------
Number of aligned timestamps:
131617

Mean absolute difference:
0.0014855352117624581

Median absolute difference:
0.0

Maximum absolute difference:
1.682116428707182

Correlation:
0.9999024826137285

P_Gaia[kW]: provided 5min vs 1min averaged to 5min
------------------------------------------------------------
Number of aligned timestamps:
131617

Mean absolute difference:
0.0029433760120828882

Median absolute difference:
0.0

Maximum absolute difference:
9.909903316894354

Correlation:
0.9992219174528717

P_Solar[kW]: provided 60min vs 5min averaged to 60min
------------------------------------------------------------
Number of aligned timestamps:
10969

Mean absolute difference:
0.05249500570431236

Median absolute difference:
0.0001051127406538066

Maximum absolute difference:
2.3782025993142555

Correlation:
0.9981984059803309

P_Gaia[kW]: provided 60min vs 5min avera

In [6]:
# ============================================================
# CELL 6 — RECONSTRUCT 60-MIN TARGETS FROM 1-SECOND DATA
# ============================================================

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Previous results showed:
#
#   1-minute -> 5-minute
#
# reproduces both P_Solar and P_Gaia extremely well.
#
# However:
#
#   5-minute -> 60-minute
#
# reproduces P_Solar well but NOT P_Gaia.
#
# At the same time, the overall distribution of P_Gaia
# in the 1-second and 60-minute datasets looks remarkably
# similar.
#
# We therefore test whether the provided 60-minute targets
# can be reconstructed directly from the 1-second data.
#
# Since the 1-second HDF5 rows are stored in shuffled daily
# blocks, we aggregate using the ACTUAL timestamp of each
# observation rather than relying on stored row order.


# ------------------------------------------------------------
# 1. DEFINE THE COMPLETE HOURLY TIME GRID
# ------------------------------------------------------------

start_time = pd.Timestamp(
    "2018-06-01 00:00:00",
    tz="UTC"
)

end_time = pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC"
)

hourly_index = pd.date_range(
    start=start_time,
    end=end_time,
    freq="60min"
)

number_of_hours = len(hourly_index)

print("Number of hourly timestamps:")
print(number_of_hours)


# ------------------------------------------------------------
# 2. HDF5 COLUMN POSITIONS
# ------------------------------------------------------------

target_columns_1sec = {
    "P_Solar[kW]": 7,
    "P_Gaia[kW]": 6,
}


# ------------------------------------------------------------
# 3. STORAGE FOR HOURLY SUMS AND COUNTS
# ------------------------------------------------------------

hourly_sums = {
    target: np.zeros(
        number_of_hours,
        dtype=np.float64
    )
    for target in target_columns_1sec
}

hourly_counts = np.zeros(
    number_of_hours,
    dtype=np.int64
)


# ------------------------------------------------------------
# 4. CONSTANTS FOR TIMESTAMP -> HOUR CONVERSION
# ------------------------------------------------------------

start_ns = start_time.value

one_hour_ns = (
    60 * 60 * 1_000_000_000
)

chunk_size = 1_000_000


# ------------------------------------------------------------
# 5. SCAN COMPLETE 1-SECOND DATASET
# ------------------------------------------------------------

with h5py.File(
    dataset_paths["1sec"],
    "r"
) as f:

    timestamps_ds = f["DATA"]["axis1"]
    values_ds = f["DATA"]["block0_values"]

    total_rows = len(timestamps_ds)

    for start in range(
        0,
        total_rows,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total_rows
        )

        # Read timestamps for this chunk.
        timestamps = timestamps_ds[
            start:end
        ]

        # Convert each timestamp into its hourly-bin number.
        #
        # Example:
        #
        #   2018-06-01 00:35 -> bin 0
        #   2018-06-01 01:10 -> bin 1
        #
        # This works even though the HDF5 rows themselves
        # are not globally chronological.
        hour_bins = (
            (timestamps - start_ns)
            // one_hour_ns
        ).astype(np.int64)


        # Count how many 1-second observations fall into
        # each hourly bin.
        hourly_counts += np.bincount(
            hour_bins,
            minlength=number_of_hours
        )


        # Accumulate target-variable sums.
        for target, column_index in (
            target_columns_1sec.items()
        ):

            values = values_ds[
                start:end,
                column_index
            ]

            hourly_sums[target] += np.bincount(
                hour_bins,
                weights=values,
                minlength=number_of_hours
            )


# ------------------------------------------------------------
# 6. CALCULATE HOURLY MEANS
# ------------------------------------------------------------

hourly_means_1sec = pd.DataFrame(
    index=hourly_index
)

hourly_means_1sec.index.name = "Timestamp"


for target in target_columns_1sec:

    hourly_means_1sec[target] = (
        hourly_sums[target]
        / hourly_counts
    )


# ------------------------------------------------------------
# 7. DISPLAY BIN-COUNT CHECK
# ------------------------------------------------------------
#
# Most complete hourly bins should contain 3600 observations.
#
# The final timestamp:
#
#     2019-09-01 00:00:00
#
# forms the final single-observation bin.

print("\nHourly observation-count summary:")

print(
    pd.Series(hourly_counts)
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 8. COMPARE RECONSTRUCTED 1-SEC -> 60-MIN
#    WITH PROVIDED 60-MIN DATA
# ------------------------------------------------------------

for target in target_variables:

    compare_target_series(

        provided=df_60min[target],

        reconstructed=hourly_means_1sec[target],

        label=(
            f"{target}: provided 60min "
            f"vs 1sec averaged to 60min"
        ),
    )

Number of hourly timestamps:
10969

Hourly observation-count summary:
1           1
3600    10968
Name: count, dtype: int64

P_Solar[kW]: provided 60min vs 1sec averaged to 60min
------------------------------------------------------------
Number of aligned timestamps:
10969

Mean absolute difference:
0.00577474754522534

Median absolute difference:
6.989867714479736e-05

Maximum absolute difference:
0.5380964994842464

Correlation:
0.9999528892140808

P_Gaia[kW]: provided 60min vs 1sec averaged to 60min
------------------------------------------------------------
Number of aligned timestamps:
10969

Mean absolute difference:
0.0

Median absolute difference:
0.0

Maximum absolute difference:
0.0

Correlation:
1.0


In [7]:
# ============================================================
# CELL 7 — RECONSTRUCT ALL 60-MIN VARIABLES FROM 1-SECOND DATA
# ============================================================

import h5py
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cell 6 showed that:
#
#     provided 60-min P_Gaia
#
# is reproduced EXACTLY by averaging the 1-second P_Gaia.
#
# We now extend that comparison to ALL 11 SOLETE variables.
#
# For every variable we compare:
#
#     provided 60-minute value
#
# against:
#
#     hourly mean calculated directly from 1-second data
#
# This helps determine whether the 60-minute dataset is
# primarily derived from the 1-second representation.
#
#
# IMPORTANT — ANGULAR VARIABLES
# ------------------------------------------------------------
#
# WIND_DIR[deg] and Azimuth[deg] are angular variables.
#
# Their arithmetic hourly means are used in THIS CELL only
# as a PROCESSING-LINEAGE diagnostic:
#
#     "Does arithmetic aggregation of the 1-second source
#      reproduce the numerical values stored in the
#      provided 60-minute file?"
#
# Arithmetic averaging of angles should NOT be interpreted
# as the preferred physical treatment of circular variables.
#
# During feature engineering, circular variables should be
# handled appropriately, for example using sine/cosine
# representations.
#
#
# GLOBAL MISSING-VALUE CHECK
# ------------------------------------------------------------
#
# Since this cell scans all ~39.5 million observations and
# all 11 variables anyway, we also verify whether the
# complete 1-second measurement matrix contains any NaNs.
#
# This avoids performing another full-file scan later.


# ============================================================
# 1. COLUMN NAMES
# ============================================================

solete_columns = list(
    df_60min.columns
)

print("Variables to compare:")

for column in solete_columns:
    print("-", column)


# ============================================================
# 2. COMPLETE HOURLY TIME GRID
# ============================================================

start_time = pd.Timestamp(
    "2018-06-01 00:00:00",
    tz="UTC"
)

end_time = pd.Timestamp(
    "2019-09-01 00:00:00",
    tz="UTC"
)

hourly_index = pd.date_range(
    start=start_time,
    end=end_time,
    freq="60min"
)

number_of_hours = len(
    hourly_index
)

print("\nNumber of hourly timestamps:")
print(number_of_hours)


# ============================================================
# 3. STORAGE FOR HOURLY AGGREGATION
# ============================================================
#
# hourly_sums_all
#     Sum of valid 1-second measurements within each hour.
#
# hourly_valid_counts
#     Number of NON-MISSING measurements for each variable
#     within each hour.
#
# We maintain counts separately for every variable so that
# this code remains correct even if some NaNs are discovered.

hourly_sums_all = np.zeros(
    (
        number_of_hours,
        len(solete_columns)
    ),
    dtype=np.float64
)

hourly_valid_counts = np.zeros(
    (
        number_of_hours,
        len(solete_columns)
    ),
    dtype=np.int64
)


# ------------------------------------------------------------
# Timestamp observation counts
# ------------------------------------------------------------
#
# This is separate from measurement-value counts.
#
# We already know from the 1-second EDA that almost every
# complete hourly interval should contain 3600 timestamps,
# except the final endpoint timestamp.

hourly_timestamp_counts = np.zeros(
    number_of_hours,
    dtype=np.int64
)


# ============================================================
# 4. GLOBAL MISSING-VALUE STORAGE
# ============================================================

missing_counts_1sec = np.zeros(
    len(solete_columns),
    dtype=np.int64
)


# ============================================================
# 5. TIMESTAMP CONSTANTS
# ============================================================

start_ns = start_time.value

one_hour_ns = (
    60
    * 60
    * 1_000_000_000
)

chunk_size = 1_000_000


# ============================================================
# 6. SCAN COMPLETE 1-SECOND DATASET
# ============================================================

with h5py.File(
    dataset_paths["1sec"],
    "r"
) as f:

    timestamps_ds = (
        f["DATA"]["axis1"]
    )

    values_ds = (
        f["DATA"]["block0_values"]
    )


    # --------------------------------------------------------
    # VERIFY COLUMN ORDER
    # --------------------------------------------------------
    #
    # The reconstruction assumes that the HDF5 column order
    # matches the column order used by the pandas datasets.
    #
    # We verify this explicitly rather than assuming it.

    hdf5_columns = [
        name.decode("utf-8")
        for name in f["DATA"]["axis0"][:]
    ]

    assert hdf5_columns == solete_columns, (
        "1-second HDF5 column order does not match "
        "the comparison DataFrame column order."
    )


    total_rows = (
        values_ds.shape[0]
    )


    # --------------------------------------------------------
    # PROCESS FILE IN CHUNKS
    # --------------------------------------------------------

    for start in range(
        0,
        total_rows,
        chunk_size
    ):

        end = min(
            start + chunk_size,
            total_rows
        )


        # ----------------------------------------------------
        # READ TIMESTAMPS
        # ----------------------------------------------------

        timestamps = timestamps_ds[
            start:end
        ]


        # Convert each timestamp into its corresponding
        # hourly-bin position.
        #
        # This remains correct even though the source HDF5
        # rows are stored in shuffled daily blocks.

        hour_bins = (
            (timestamps - start_ns)
            // one_hour_ns
        ).astype(np.int64)


        # Count timestamp observations in each hourly bin.

        hourly_timestamp_counts += np.bincount(
            hour_bins,
            minlength=number_of_hours
        )


        # ----------------------------------------------------
        # READ ALL 11 MEASUREMENT VARIABLES
        # ----------------------------------------------------

        values = values_ds[
            start:end,
            :
        ]


        # ----------------------------------------------------
        # COUNT GLOBAL MISSING VALUES
        # ----------------------------------------------------

        missing_counts_1sec += (
            np.isnan(values)
            .sum(axis=0)
        )


        # ----------------------------------------------------
        # ACCUMULATE EACH VARIABLE INTO HOURLY BINS
        # ----------------------------------------------------
        #
        # We explicitly exclude NaNs if any exist.
        #
        # This ensures that a missing 1-second measurement
        # would not turn an entire reconstructed hourly
        # value into NaN.

        for column_index in range(
            len(solete_columns)
        ):

            column_values = values[
                :,
                column_index
            ]

            valid_mask = ~np.isnan(
                column_values
            )


            # Sum valid measurements within each hour.

            hourly_sums_all[
                :,
                column_index
            ] += np.bincount(
                hour_bins[valid_mask],
                weights=column_values[
                    valid_mask
                ],
                minlength=number_of_hours
            )


            # Count valid measurements within each hour.

            hourly_valid_counts[
                :,
                column_index
            ] += np.bincount(
                hour_bins[valid_mask],
                minlength=number_of_hours
            )


# ============================================================
# 7. GLOBAL 1-SECOND MISSING-VALUE SUMMARY
# ============================================================

missing_values_1sec = pd.Series(
    missing_counts_1sec,
    index=solete_columns,
    name="missing_values"
)

print(
    "\nMissing values across the complete "
    "1-second dataset:"
)

display(
    missing_values_1sec
)

print("\nTotal missing values:")
print(
    missing_values_1sec.sum()
)


# ============================================================
# 8. VERIFY HOURLY TIMESTAMP COUNTS
# ============================================================

print("\nHourly timestamp-count distribution:")

print(
    pd.Series(
        hourly_timestamp_counts
    )
    .value_counts()
    .sort_index()
)


# ============================================================
# 9. CALCULATE HOURLY MEANS
# ============================================================

hourly_means_all_1sec = np.divide(

    hourly_sums_all,

    hourly_valid_counts,

    # If a variable somehow had no valid values inside
    # a particular hour, return NaN rather than divide
    # by zero.
    out=np.full_like(
        hourly_sums_all,
        np.nan,
        dtype=np.float64
    ),

    where=(
        hourly_valid_counts > 0
    )
)


# Create chronological hourly DataFrame.

df_1sec_to_60min = pd.DataFrame(
    hourly_means_all_1sec,
    columns=solete_columns,
    index=hourly_index
)

df_1sec_to_60min.index.name = (
    "Timestamp"
)


# ============================================================
# 10. COMPARE PROVIDED VS RECONSTRUCTED VARIABLES
# ============================================================

comparison_results = []


for variable in solete_columns:

    comparison = pd.concat(
        [
            df_60min[
                variable
            ].rename(
                "provided"
            ),

            df_1sec_to_60min[
                variable
            ].rename(
                "reconstructed"
            ),
        ],
        axis=1,
        join="inner",
    ).dropna()


    # --------------------------------------------------------
    # ABSOLUTE DIFFERENCE
    # --------------------------------------------------------

    absolute_difference = (
        comparison["provided"]
        -
        comparison["reconstructed"]
    ).abs()


    # --------------------------------------------------------
    # CORRELATION
    # --------------------------------------------------------
    #
    # For angular variables such as WIND_DIR and Azimuth,
    # this correlation is only a numerical processing-lineage
    # diagnostic and should not be interpreted as a physical
    # circular association measure.

    correlation = (
        comparison[
            [
                "provided",
                "reconstructed"
            ]
        ]
        .corr()
        .iloc[0, 1]
    )


    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------

    comparison_results.append({

        "variable":
            variable,

        "mean_abs_difference":
            absolute_difference.mean(),

        "median_abs_difference":
            absolute_difference.median(),

        "max_abs_difference":
            absolute_difference.max(),

        "correlation":
            correlation,
    })


# ============================================================
# 11. DISPLAY FINAL COMPARISON TABLE
# ============================================================

all_variable_comparison = (
    pd.DataFrame(
        comparison_results
    )
    .set_index(
        "variable"
    )
)

print(
    "\nProvided 60-minute data vs "
    "1-second data averaged to 60 minutes:"
)

all_variable_comparison

Variables to compare:
- TEMPERATURE[degC]
- HUMIDITY[%]
- WIND_SPEED[m1s]
- WIND_DIR[deg]
- GHI[kW1m2]
- POA Irr[kW1m2]
- P_Gaia[kW]
- P_Solar[kW]
- Pressure[mbar]
- Azimuth[deg]
- Elevation[deg]

Number of hourly timestamps:
10969

Missing values across the complete 1-second dataset:


TEMPERATURE[degC]    0
HUMIDITY[%]          0
WIND_SPEED[m1s]      0
WIND_DIR[deg]        0
GHI[kW1m2]           0
POA Irr[kW1m2]       0
P_Gaia[kW]           0
P_Solar[kW]          0
Pressure[mbar]       0
Azimuth[deg]         0
Elevation[deg]       0
Name: missing_values, dtype: int64


Total missing values:
0

Hourly timestamp-count distribution:
1           1
3600    10968
Name: count, dtype: int64

Provided 60-minute data vs 1-second data averaged to 60 minutes:


,mean_abs_difference,median_abs_difference,max_abs_difference,correlation
variable,,,,
TEMPERATURE[degC],0.143595,0.00000,22.299901,0.982781
HUMIDITY[%],0.013121,0.00000,1.799750,0.844891
WIND_SPEED[m1s],0.050934,0.00000,12.099708,0.971931
WIND_DIR[deg],3.187205,0.00000,405.913272,0.932551
GHI[kW1m2],0.000026,0.00000,0.130932,0.999979
POA Irr[kW1m2],0.000024,0.00000,0.068158,0.999990
P_Gaia[kW],0.000000,0.00000,0.000000,1.000000
P_Solar[kW],0.005775,0.00007,0.538096,0.999953
Pressure[mbar],41.651295,0.00000,1999.444444,0.228387


In [8]:
# ============================================================
# CELL 8 — RECONSTRUCT ALL 5-MIN VARIABLES FROM 1-MIN DATA
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Earlier comparisons showed that:
#
#     1-minute -> 5-minute
#
# reproduces both renewable-energy target variables extremely
# closely.
#
# We now extend that comparison to ALL 11 SOLETE variables.
#
# The purpose is to determine whether the provided 5-minute
# dataset belongs to the same processing lineage as the
# provided 1-minute dataset.
#
#
# IMPORTANT — ANGULAR VARIABLES
# ------------------------------------------------------------
#
# WIND_DIR[deg] and Azimuth[deg] are angular variables.
#
# Arithmetic averaging is used here ONLY as a
# processing-lineage diagnostic:
#
#     "Does pandas arithmetic resampling of the 1-minute
#      values reproduce the values stored in the provided
#      5-minute file?"
#
# This should NOT be interpreted as the preferred physical
# treatment of circular variables during feature engineering.
#
# Later modelling will use appropriate circular
# representations where required.


# ============================================================
# 1. RECONSTRUCT 5-MINUTE DATA FROM 1-MINUTE DATA
# ============================================================

df_1min_to_5min = (
    df_1min[
        solete_columns
    ]
    .resample(
        "5min"
    )
    .mean()
)


# ============================================================
# 2. BASIC RECONSTRUCTION CHECK
# ============================================================

print(
    "Provided 5-minute shape:"
)

print(
    df_5min.shape
)

print(
    "\nReconstructed 5-minute shape:"
)

print(
    df_1min_to_5min.shape
)

print(
    "\nProvided 5-minute date range:"
)

print(
    df_5min.index.min(),
    "to",
    df_5min.index.max()
)

print(
    "\nReconstructed 5-minute date range:"
)

print(
    df_1min_to_5min.index.min(),
    "to",
    df_1min_to_5min.index.max()
)


# ============================================================
# 3. STORAGE FOR VARIABLE-BY-VARIABLE RESULTS
# ============================================================

comparison_results_1min_5min = []


# ============================================================
# 4. COMPARE EVERY VARIABLE
# ============================================================

for variable in solete_columns:

    # --------------------------------------------------------
    # ALIGN PROVIDED AND RECONSTRUCTED DATA
    # --------------------------------------------------------

    comparison = pd.concat(
        [
            df_5min[
                variable
            ].rename(
                "provided"
            ),

            df_1min_to_5min[
                variable
            ].rename(
                "reconstructed"
            ),
        ],
        axis=1,
        join="inner",
    ).dropna()


    # --------------------------------------------------------
    # ABSOLUTE DIFFERENCE
    # --------------------------------------------------------

    absolute_difference = (
        comparison["provided"]
        -
        comparison["reconstructed"]
    ).abs()


    # --------------------------------------------------------
    # PEARSON CORRELATION
    # --------------------------------------------------------
    #
    # This measures numerical similarity between the two
    # representations.
    #
    # For angular variables, it is NOT being used as a
    # physical circular correlation measure.

    correlation = (
        comparison[
            [
                "provided",
                "reconstructed"
            ]
        ]
        .corr()
        .iloc[0, 1]
    )


    # --------------------------------------------------------
    # STRICT NUMERICAL MATCH CHECK
    # --------------------------------------------------------
    #
    # Floating-point averages may differ by extremely small
    # numerical amounts even when they originate from the
    # same processing operation.
    #
    # Therefore, instead of using:
    #
    #     provided == reconstructed
    #
    # we use np.isclose() with a deliberately strict
    # numerical tolerance.
    #
    # IMPORTANT:
    #
    # This percentage should be interpreted as:
    #
    #     "strict numerical agreement"
    #
    # rather than general statistical similarity.

    strict_numeric_matches = np.isclose(
        comparison["provided"],
        comparison["reconstructed"],
        rtol=1e-9,
        atol=1e-9
    )


    strict_numeric_match_percent = (
        strict_numeric_matches.sum()
        / len(comparison)
        * 100
    )


    # --------------------------------------------------------
    # STORE RESULT
    # --------------------------------------------------------

    comparison_results_1min_5min.append({

        "variable":
            variable,

        "mean_abs_difference":
            absolute_difference.mean(),

        "median_abs_difference":
            absolute_difference.median(),

        "max_abs_difference":
            absolute_difference.max(),

        "correlation":
            correlation,

        "strict_numeric_match_percent":
            strict_numeric_match_percent,
    })


# ============================================================
# 5. CREATE FINAL COMPARISON TABLE
# ============================================================

all_variable_comparison_1min_5min = (
    pd.DataFrame(
        comparison_results_1min_5min
    )
    .set_index(
        "variable"
    )
)


# ============================================================
# 6. DISPLAY RESULT
# ============================================================

print(
    "Provided 5-minute data vs "
    "1-minute data averaged to 5 minutes:"
)

all_variable_comparison_1min_5min

Provided 5-minute shape:
(131617, 11)

Reconstructed 5-minute shape:
(131617, 11)

Provided 5-minute date range:
2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00

Reconstructed 5-minute date range:
2018-06-01 00:00:00+00:00 to 2019-09-01 00:00:00+00:00
Provided 5-minute data vs 1-minute data averaged to 5 minutes:


,mean_abs_difference,median_abs_difference,max_abs_difference,correlation,strict_numeric_match_percent
variable,,,,,
TEMPERATURE[degC],9.579197e-03,0.0,1.783978e+01,0.999048,99.653540
HUMIDITY[%],8.761809e-04,0.0,8.000000e-01,0.988202,99.836647
WIND_SPEED[m1s],3.451761e-03,0.0,9.680528e+00,0.998443,99.576802
WIND_DIR[deg],2.135106e-01,0.0,2.849743e+02,0.996198,99.577562
GHI[kW1m2],2.959082e-06,0.0,1.303890e-01,0.999997,99.915664
POA Irr[kW1m2],2.567012e-06,0.0,1.554173e-01,0.999998,99.886033
P_Gaia[kW],2.943376e-03,0.0,9.909903e+00,0.999222,99.880714
P_Solar[kW],1.485535e-03,0.0,1.682116e+00,0.999902,79.527721
Pressure[mbar],2.771678e+00,0.0,8.000000e+02,0.859642,99.653540


## Cross-Resolution Processing-Lineage Findings

The reconstruction experiments show that the four SOLETE resolutions
should not be interpreted as one simple sequential resampling hierarchy.

### 1-Minute to 5-Minute Relationship

A 5-minute dataset reconstructed by averaging the provided 1-minute
observations closely reproduces the provided SOLETE 5-minute dataset.

For most variables:

- correlations are above 0.99,
- median absolute differences are zero,
- and more than 99% of observations are numerically identical or nearly
  identical.

`Azimuth[deg]` and `Elevation[deg]` are reproduced essentially exactly.

`P_Solar[kW]` also shows extremely high agreement
(correlation approximately 0.9999), although small floating-point or
processing differences reduce the percentage of strictly near-identical
observations under a very tight numerical tolerance.

Pressure is a notable exception because of the systematic daily-boundary
artifact identified during EDA.

These results indicate that the **1-minute and 5-minute datasets belong
to the same general processing lineage**.

### 1-Second to 60-Minute Relationship

Direct hourly aggregation of the 1-second dataset reproduces several
provided 60-minute variables extremely closely.

Most importantly:

- `P_Gaia[kW]` is reproduced **exactly**:
  - mean absolute difference = 0
  - maximum absolute difference = 0
  - correlation = 1.0
- `Azimuth[deg]` is reproduced exactly.
- `Elevation[deg]` is reproduced exactly.
- GHI, POA irradiance, and solar power are also reproduced with very
  high agreement.

This explains several unusual characteristics previously observed in the
60-minute dataset.

The provided 60-minute `P_Gaia[kW]` is therefore not equivalent to a
simple hourly average of the provided 5-minute wind-power series.
Instead, it is exactly consistent with hourly aggregation of the
1-second wind-power representation.

### Implication

The evidence suggests two practical SOLETE processing branches:

1. **1-second → 60-minute**
2. **1-minute → 5-minute**

This distinction is important because variables with the same names do
not necessarily have equivalent distributions or behaviour across all
four resolution files.

Therefore, temporal resolution must be selected based not only on sample
frequency but also on the underlying variable representation and
processing lineage.

### Current Candidate Resolutions

Based on the analysis so far:

- **1-second** provides maximum temporal detail but is very large,
  stored in shuffled daily blocks, and contains extremely sparse values
  for several important variables.
- **60-minute** is computationally simple but inherits several unusual
  characteristics from the 1-second representation and provides only
  10,969 observations.
- **1-minute** is structurally clean and retains high temporal detail.
- **5-minute** is structurally clean, preserves the main physical
  relationships of the 1-minute data, and substantially reduces the
  computational burden.

Therefore, the strongest candidates for the main forecasting experiments
are currently the **1-minute and 5-minute resolutions**.

A final selection will be made after directly comparing their forecasting
information content, computational cost, and practical suitability.

In [9]:
# ============================================================
# CELL 9 — 1-MINUTE VS 5-MINUTE FORECASTING SCORECARD
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# Cross-resolution analysis has narrowed the strongest
# forecasting candidates to:
#
#     1-minute
#     5-minute
#
# Both datasets:
#
#   - are chronologically ordered,
#   - contain no missing values,
#   - contain no duplicate timestamps,
#   - preserve the main solar/wind relationships,
#   - and belong to the same processing lineage.
#
# We now compare them from a PRACTICAL forecasting perspective.
#
# This is not yet a model-performance comparison.
# It is a dataset-selection scorecard based on the evidence
# established during EDA and cross-resolution analysis.


# ------------------------------------------------------------
# 1. BASIC COMPUTATIONAL SIZE
# ------------------------------------------------------------

rows_1min = len(df_1min)
rows_5min = len(df_5min)

size_ratio = (
    rows_1min
    / rows_5min
)


# ------------------------------------------------------------
# 2. BUILD SCORECARD
# ------------------------------------------------------------

candidate_scorecard = pd.DataFrame(
    {
        "criterion": [
            "Number of observations",
            "Relative dataset size",
            "Sampling interval",
            "Chronological source order",
            "Missing values",
            "Duplicate timestamps",
            "Complete regular grid",
            "Solar relationships preserved",
            "Wind relationships preserved",
            "Short-term variability retained",
            "Computational burden",
            "Sequence length for same forecast horizon",
            "Ease of experimentation",
        ],

        "1-minute": [
            f"{rows_1min:,}",
            f"{size_ratio:.1f} × 5-minute size",
            "1 minute",
            "Yes",
            "0",
            "0",
            "Yes",
            "Yes",
            "Yes",
            "Higher",
            "Higher",
            "5 × longer",
            "Moderate",
        ],

        "5-minute": [
            f"{rows_5min:,}",
            "1.0 ×",
            "5 minutes",
            "Yes",
            "0",
            "0",
            "Yes",
            "Yes",
            "Yes",
            "Moderate",
            "Lower",
            "1 ×",
            "High",
        ],
    }
)


# ------------------------------------------------------------
# 3. DISPLAY SCORECARD
# ------------------------------------------------------------

candidate_scorecard

,criterion,1-minute,5-minute
0,Number of observations,"658,081","131,617"
1,Relative dataset size,5.0 × 5-minute size,1.0 ×
2,Sampling interval,1 minute,5 minutes
3,Chronological source order,Yes,Yes
4,Missing values,0,0
5,Duplicate timestamps,0,0
6,Complete regular grid,Yes,Yes
7,Solar relationships preserved,Yes,Yes
8,Wind relationships preserved,Yes,Yes
9,Short-term variability retained,Higher,Moderate


In [10]:
# ============================================================
# CELL 10 — SEQUENCE LENGTHS FOR EQUAL PHYSICAL TIME WINDOWS
# ============================================================

import pandas as pd


# ------------------------------------------------------------
# PURPOSE
# ------------------------------------------------------------
#
# The 1-minute dataset contains five times as many observations
# as the 5-minute dataset.
#
# For time-series forecasting, this also means that the same
# PHYSICAL history window requires five times as many model
# time steps.
#
# Example:
#
#     Previous 24 hours
#
#     1-minute data -> 1440 time steps
#     5-minute data ->  288 time steps
#
# Longer sequences generally require more memory and
# computation, particularly for recurrent models such as
# LSTM / BiLSTM.
#
# The time windows below are illustrative comparisons only.
# They do NOT define our final forecasting horizon.


# ------------------------------------------------------------
# SAMPLING INTERVALS
# ------------------------------------------------------------

sampling_minutes = {
    "1-minute": 1,
    "5-minute": 5,
}


# ------------------------------------------------------------
# PHYSICAL WINDOWS TO COMPARE
# ------------------------------------------------------------

physical_windows = {
    "1 hour": 60,
    "6 hours": 6 * 60,
    "12 hours": 12 * 60,
    "24 hours": 24 * 60,
    "48 hours": 48 * 60,
}


# ------------------------------------------------------------
# BUILD COMPARISON TABLE
# ------------------------------------------------------------

sequence_rows = []


for window_name, window_minutes in physical_windows.items():

    row = {
        "physical_window": window_name
    }

    for resolution, interval_minutes in sampling_minutes.items():

        number_of_steps = (
            window_minutes
            // interval_minutes
        )

        row[resolution] = number_of_steps

    # Confirm the relative sequence-length difference.
    row["1min_to_5min_ratio"] = (
        row["1-minute"]
        / row["5-minute"]
    )

    sequence_rows.append(row)


sequence_length_comparison = (
    pd.DataFrame(sequence_rows)
    .set_index("physical_window")
)


# ------------------------------------------------------------
# DISPLAY RESULT
# ------------------------------------------------------------

sequence_length_comparison

,1-minute,5-minute,1min_to_5min_ratio
physical_window,,,
1 hour,60,12,5.0
6 hours,360,72,5.0
12 hours,720,144,5.0
24 hours,1440,288,5.0
48 hours,2880,576,5.0


## Final Temporal-Resolution Selection

The cross-resolution analysis indicates that the four SOLETE files should
not be treated as a single uniform resampling hierarchy.

Two practical processing lineages were identified:

1. **1-second → 60-minute**
2. **1-minute → 5-minute**

The strongest candidates for the main forecasting experiments are therefore
the **1-minute** and **5-minute** datasets.

### Why the 60-Minute Resolution Is Not Selected

The 60-minute dataset contains only **10,969 observations**, substantially
reducing the number of training samples available for machine-learning and
deep-learning experiments.

More importantly, several variables inherit unusual characteristics from
the 1-second representation.

For example:

- `P_Gaia[kW]` in the 60-minute dataset is reproduced exactly by hourly
  averaging of the 1-second wind-power series;
- the 60-minute `P_Gaia[kW]` distribution differs substantially from the
  1-minute and 5-minute wind-power distributions;
- `Azimuth[deg]` and `Elevation[deg]` also reproduce the sparse 1-second
  representation;
- pressure retains substantial source-level artifacts.

The 60-minute dataset is therefore useful for cross-resolution analysis,
but it is not the preferred representation for the main multivariable
forecasting experiments.

### Why the 1-Second Resolution Is Not Selected

The 1-second dataset provides the highest temporal detail and contains a
complete and unique one-second timestamp grid.

However, it also has several practical limitations:

- approximately **39.5 million observations**;
- approximately **3.53 GB** of HDF5 data;
- shuffled daily-sized storage blocks;
- extremely sparse non-zero values for variables such as:
  - `P_Gaia[kW]`,
  - `Azimuth[deg]`,
  - `Elevation[deg]`;
- substantially higher memory and computational requirements.

The 1-second dataset remains valuable for source-level diagnostics but is
not well suited to the main forecasting workflow on the available
computational environment.

### 1-Minute vs 5-Minute Resolution

Both the 1-minute and 5-minute datasets are structurally clean:

- no missing measurements;
- no duplicate timestamps;
- chronological source ordering;
- complete regular temporal grids;
- consistent temporal coverage;
- similar solar and wind relationships.

Furthermore, direct reconstruction showed that averaging the 1-minute
dataset to 5-minute intervals reproduces the provided 5-minute dataset
extremely closely for most variables.

This demonstrates that the 5-minute representation retains most of the
information available in the 1-minute representation while substantially
reducing dataset size.

### Computational Trade-Off

The 1-minute dataset contains:

- **658,081 observations**

while the 5-minute dataset contains:

- **131,617 observations**

Therefore, the 1-minute dataset is approximately **5 times larger**.

The same difference appears in forecasting sequence lengths.

For example, a 24-hour historical input window requires:

- **1-minute data:** 1,440 time steps
- **5-minute data:** 288 time steps

Similarly, a 48-hour historical input window requires:

- **1-minute data:** 2,880 time steps
- **5-minute data:** 576 time steps

This difference is particularly important for recurrent and sequence-based
models such as LSTM and BiLSTM, where longer input sequences increase
training time, memory usage, and model complexity.

### Selected Main Resolution

Based on the combined evidence, the **5-minute SOLETE dataset is selected
as the primary temporal resolution for the main forecasting experiments**.

The selection is based on its balance of:

- sufficient short-term temporal detail;
- clean timestamp structure;
- preservation of the important physical relationships present at
  1-minute resolution;
- substantially lower computational cost;
- manageable sequence lengths;
- sufficient number of observations for model training and evaluation;
- and better suitability for repeated experimentation and hyperparameter
  tuning.

### Role of the 1-Minute Dataset

The 1-minute dataset will not be discarded.

It can be retained as a **secondary or sensitivity-resolution dataset**.

If required later, selected experiments may be repeated at 1-minute
resolution to investigate whether the additional short-term temporal detail
produces a meaningful forecasting improvement over the 5-minute model.

This provides a stronger experimental design than selecting a resolution
only on the basis of convenience.

### Final Resolution Strategy

The project will therefore use:

**Primary modelling resolution**
- `SOLETE_Pombo_5min.h5`

**Secondary / sensitivity resolution**
- `SOLETE_Pombo_1min.h5`

**Cross-resolution / diagnostic datasets**
- `SOLETE_Pombo_60min.h5`
- `SOLETE_Pombo_1sec.h5`

The next stage of the project will begin with preprocessing and feature
engineering of the selected 5-minute dataset.